# 8 — YOLO: detekcija objekata

**Četvrtak, 13:00.** Prvi dio popodnevnog bloka.

YOLO **nije** generativni model — to je *diskriminativni* detektor treniran za
jedan zadatak. Zato je ovdje: da vidimo razliku između uskog modela koji radi
jednu stvar pouzdano i općeg modela kojem zadatak *opišemo*.

Na kraju ćemo YOLO zapakirati u funkciju `detect()`. **Tu funkciju ćemo danas
u 16h dati AI agentu kao alat**, pa nemojte zatvoriti ovaj notebook.

Sve radi na **CPU-u**. GPU nije potreban i ne tražite ga.

In [ ]:
# --- SETUP: pokreni ovo prvo ---  [lares-setup-v1]
# Radi i u Colabu i lokalno. Sigurno je pokrenuti vise puta.
import os, sys, subprocess
REPO = "ai_bootcamp_foundations"
if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/hrvojenovak/{REPO}.git"],
                       cwd="/content", check=True)
    os.chdir(f"/content/{REPO}/notebooks")
print("cwd:", os.getcwd(), "| data ok:", os.path.isdir("../data"))

## 1. Instalacija

Traje ~30 s. Ako Colab zatraži **Restart runtime**, restartirajte i pokrenite
sve ćelije ispočetka — SETUP ćelija je napisana tako da to preživi.

In [ ]:
%pip install -q ultralytics

import os
os.environ["YOLO_VERBOSE"] = "False"

# Ultralytics po defaultu salje anonimnu telemetriju. Iskljucujemo je -
# javne slike nisu problem, ali navika je dobra.
from ultralytics import settings
settings.update({"sync": False})

import ultralytics, torch
torch.set_num_threads(2)          # Colab free ima 2 vCPU
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("CUDA dostupan:", torch.cuda.is_available(), " <- ne treba nam")

## 2. Model

`yolo11n` je "nano" varijanta: **5.6 MB**, ~2.6M parametara. Postoje i veće
(`s`, `m`, `l`, `x`) — `yolo11s` je 19 MB i oko 2× sporija.

Za demo je nano više nego dovoljna.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")        # tezine se skidaju automatski (~5.6 MB)

n_params = sum(p.numel() for p in model.model.parameters())
print(f"parametara: {n_params/1e6:.1f}M")
print(f"klasa koje poznaje: {len(model.names)}")
print(f"primjeri: {[model.names[i] for i in [0, 2, 5, 15, 39, 63]]}")

## 3. Prvi inference + mjerenje

Mjerimo namjerno. Poanta koju treba zapamtiti: **inference i trening su
sasvim različiti resursni problemi.** Ljudi ih spajaju i onda misle da im
za sve treba GPU.

In [ ]:
import time, urllib.request

IMG = "bus.jpg"
if not os.path.exists(IMG):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/ultralytics/ultralytics"
        "/main/ultralytics/assets/bus.jpg", IMG)

model(IMG, verbose=False)                     # warmup (prvi poziv je sporiji)

t = time.time()
results = model(IMG, verbose=False)
dt = (time.time() - t) * 1000

print(f"inference: {dt:.0f} ms na CPU-u\n")
for box in results[0].boxes:
    print(f"  {model.names[int(box.cls)]:12s} pouzdanost {float(box.conf):.2f}")

## 4. Kako to izgleda

`results[0].plot()` nacrta okvire. Vraća **BGR** niz (OpenCV konvencija), pa
kanale treba obrnuti prije prikaza — klasična zamka.

In [ ]:
import matplotlib.pyplot as plt

annotated = results[0].plot()[:, :, ::-1]     # BGR -> RGB

plt.figure(figsize=(7, 9))
plt.imshow(annotated)
plt.axis("off")
plt.title(f"yolo11n — {len(results[0].boxes)} detekcija, {dt:.0f} ms (CPU)")
plt.show()

## 5. `detect()` — alat za agenta

Ovo je najvažnija ćelija u notebooku.

Do sad smo YOLO koristili interaktivno. Sad ga zatvaramo u funkciju s **čistim
potpisom**: `detect(path) -> list[dict]`. Ulaz je putanja, izlaz je obični
Python/JSON.

Zašto to radimo: **agentu je alat bilo što s definiranim interfaceom**, ne samo
tekst. Kad u 16h agent dobije `detect` uz `run_python`, moći će "vidjeti"
slike — a mi nismo napisali ni jednu novu liniju modela.

In [ ]:
def detect(path: str, conf: float = 0.25) -> list[dict]:
    """Detektiraj objekte na slici.

    Args:
        path: putanja do slike
        conf: minimalna pouzdanost (0-1)

    Returns:
        Lista detekcija: label, confidence, box [x1, y1, x2, y2].
    """
    if not os.path.exists(path):
        return [{"error": f"nema datoteke: {path}"}]

    r = model(path, conf=conf, verbose=False)[0]
    return [
        {
            "label": model.names[int(b.cls)],
            "confidence": round(float(b.conf), 3),
            "box": [round(v) for v in b.xyxy[0].tolist()],
        }
        for b in r.boxes
    ]


import json
print(json.dumps(detect(IMG), indent=1))

## 6. Vježba: gdje YOLO prestaje

Učitajte **svoju** sliku — najbolje nešto iz vaše domene (nadzemni vod, PV
panel, trafostanica, izolator).

Očekujte da neće raditi. YOLO poznaje **80 COCO klasa** i to je sve što dobijete
bez fine-tuninga. PV panel će prijaviti kao `tv`, ili neće prijaviti ništa.

**To nije bug, to je poanta.** Uski model radi točno ono za što je treniran i
ništa izvan toga. Držite tu razliku u glavi kad za sat vremena vidite opći
model kojem zadatak samo *opišete*.

In [ ]:
# Colab: otvara dialog za odabir datoteke
if "google.colab" in sys.modules:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        print(f"\n=== {name} ===")
        found = detect(name)
        print(json.dumps(found, indent=1) if found else "  NISTA DETEKTIRANO")
else:
    print("Lokalno: detect('putanja/do/slike.jpg')")

## 7. Za zapamtiti

**80 klasa, i to je sve.** Za svoju domenu treba fine-tuning na označenim
podacima — a označavanje je posao, ne detalj.

**GPU ne treba za inference.** Izmjereno: ~100 ms po slici na CPU-u.
GPU je za *trening* i za *skaliranje*, ne za učenje.

**Licenca: `ultralytics` je AGPL-3.0.** To je copyleft koji se proteže i na
mrežno korištenje. Za tečaj nebitno, **za vaše firme može biti problem.**
Alternative s permisivnijim licencama: torchvision detektori (BSD),
YOLOX (Apache 2.0), RT-DETR.

**`detect()` ostaje.** U 16h je agent dobiva kao alat.